## 1.Executive Summary

This project analyzes a simulated payments platform to uncover user behavior, payment success patterns, and funnel performance using SQL, pandas, and A/B experiment validation.

Key findings:
- Payment failures increase during late hours
- Mobile users have lower conversion rates than web
- Repeat users drive higher engagement but not higher ticket size
- Experiment variant B delivered a 4.42 pp lift in page→success conversion with strong statistical significance

Recommendations:
- Improve mobile checkout experience
- Add fallback/retry mechanisms for high-failure hours
- Optimize payment flow using experiment-driven tests

## 2.Business Context

This project simulates a payments platform (similar to Cashfree),
analyzing both user-side (B2C) and merchant-side (B2B) behavior.

Goal:
- Understand user payment behavior
- Identify drop-offs in payment funnel
- Analyze failure patterns
- Provide actionable business recommendations

## 3.Data Understanding

We have 5 datasets:
- users
- transactions
- events
- merchants
- experiments

Goal: To Understand structure, columns, and relationships

In [103]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("payflow.db")

users = pd.read_csv("../data/users.csv")
transactions = pd.read_csv("../data/transactions.csv")
events = pd.read_csv("../data/events.csv")
merchants = pd.read_csv("../data/merchants.csv")
experiments = pd.read_csv("../data/experiments.csv")

users.to_sql("users", conn, if_exists="replace", index=False)
transactions.to_sql("transactions", conn, if_exists="replace", index=False)
events.to_sql("events", conn, if_exists="replace", index=False)
merchants.to_sql("merchants", conn, if_exists="replace", index=False)
experiments.to_sql("experiments", conn, if_exists="replace", index=False)

print("✅ Data loaded")

✅ Data loaded


In [105]:
## Data:

transactions_df = run_query("SELECT * FROM transactions")
users_df = run_query("SELECT * FROM users")
events_df = run_query("SELECT * FROM events")
merchants_df = run_query("select * from merchants ")
experiments_df = run_query("select * from experiments")

In [104]:
## HELPER FUNCTION

def run_query(query):
    return pd.read_sql(query, conn)

In [106]:
transactions_df.head(5)

,txn_id,user_id,merchant_id,merchant_category,amount,payment_method,status,failure_reason,is_repeat_user,timestamp
0,0,41190,1244,gaming,1010.26,UPI,success,None,1,2026-02-19 18:46:00
1,1,47526,948,fintech,924.03,card,success,None,1,2026-03-15 14:38:12
2,2,38158,3249,ecommerce,1966.02,UPI,success,None,0,2026-01-14 13:27:36
3,3,30603,1055,gaming,5197.53,card,failed,network_error,1,2026-02-19 05:47:32
4,4,49642,551,fintech,974.97,UPI,success,None,0,2026-02-16 21:50:13


In [107]:
users_df.head(5)


,user_id,signup_date,device,city,is_active,cohort_month
0,0,2025-09-18,android,East Christopher,1,2025-12
1,1,2025-08-30,android,Cameronside,1,2025-09
2,2,2025-08-30,android,Hodgeborough,1,2026-01
3,3,2025-04-18,android,Lake Ryan,1,2025-06
4,4,2025-05-26,android,Williamborough,0,2025-05


In [108]:
events_df.head()


,user_id,device,event,variant,timestamp
0,9121,android,page_view,B,2026-02-12 23:46:02
1,9121,android,initiate_payment,B,2026-02-21 04:56:36
2,9121,android,enter_otp,B,2026-01-16 05:20:14
3,36888,android,page_view,B,2026-01-30 08:14:27
4,36888,android,initiate_payment,B,2026-03-22 16:48:43


In [109]:
merchants_df.head()

,merchant_id,category,onboarding_days_ago,pricing_plan,is_active
0,0,ecommerce,43,premium,1
1,1,ecommerce,102,basic,0
2,2,fintech,288,premium,1
3,3,gaming,226,basic,1
4,4,fintech,310,premium,1


## 4.Data Quality Checks

ROW COUNT CHECK:

In [110]:
transactions_df.shape


(200000, 10)

In [111]:
users_df.shape


(50000, 6)

In [112]:
events_df.shape

(288331, 5)

### We have ~200k transactions, ~50k users, and event-level data capturing user journey.

In [ ]:
#NULL VALUE 

transactions_df.isnull().sum()

txn_id                    0
user_id                   0
merchant_id               0
merchant_category         0
amount                    0
payment_method            0
status                    0
failure_reason       180819
is_repeat_user            0
timestamp                 0
dtype: int64

In [153]:
events_df.isnull().sum()

user_id      0
device       0
event        0
variant      0
timestamp    0
dtype: int64

In [154]:
users_df.isnull().sum()

user_id         0
signup_date     0
device          0
city            0
is_active       0
cohort_month    0
dtype: int64

No significant null values observed in key columns.

In [113]:
# DUPLICATES

transactions_df['txn_id'].duplicated().sum()

## there is no duplicates

0

In [155]:
users_df['user_id'].duplicated().sum()

0

In [156]:
users_df['user_id'].duplicated().sum()

0

No duplicate records found. All primary keys are unique.

RANGE / OUTLIER CHECK :

In [157]:
## Amount distribution:

transactions_df['amount'].describe()

count    200000.000000
mean       2296.903386
std        1848.097725
min          30.010000
25%         913.297500
50%        1762.265000
75%        3144.935000
max        7999.660000
Name: amount, dtype: float64

#### **No major anomalies found in transactions data. Values fall within expected ranges.


In [158]:
## negative values:

transactions_df[transactions_df['amount'] < 0]

,txn_id,user_id,merchant_id,merchant_category,amount,payment_method,status,failure_reason,is_repeat_user,timestamp,hour


##### UNIVARIATE ANALYSIS :

## 5.EDA

##### UNIVARIATE ANALYSIS :

In [116]:
#payment_method

transactions_df['payment_method'].value_counts()

payment_method
UPI       119746
card       60273
wallet     19981
Name: count, dtype: int64

In [117]:
#status

transactions_df['status'].value_counts(normalize=True)

status
success    0.904095
failed     0.095905
Name: proportion, dtype: float64

In [118]:
#Amount

transactions_df['amount'].describe()

count    200000.000000
mean       2296.903386
std        1848.097725
min          30.010000
25%         913.297500
50%        1762.265000
75%        3144.935000
max        7999.660000
Name: amount, dtype: float64

🧠 Analysis :

Which method is dominant? - UPI

What’s success vs failure ratio? - 9.4/1 (So the success-to-failure ratio is about 9.4 successes for every 1 failure.)

#####  BIVARIATE ANALYSIS:

In [119]:
## Success rate by payment method

transactions_df.groupby('payment_method')['status'].value_counts(normalize=True)

payment_method  status 
UPI             success    0.908790
                failed     0.091210
card            success    0.879366
                failed     0.120634
wallet          success    0.950553
                failed     0.049447
Name: proportion, dtype: float64

In [120]:
## Revenue by merchant category

transactions_df.groupby('merchant_category')['amount'].sum().sort_values(ascending=False)

merchant_category
gaming       2.046057e+08
fintech      1.276956e+08
ecommerce    7.640785e+07
edtech       5.067151e+07
Name: amount, dtype: float64

In [121]:
## Repeat vs new users

transactions_df.groupby('is_repeat_user')['amount'].agg(['count','mean'])

,count,mean
is_repeat_user,,
0,80176,2302.885104
1,119824,2292.900930


🧠 Analysis :

Who brings more revenue? - gaming industry

Which segment behaves differently? - 

*is_repeat_user = 1 has far more transactions: 119,824 vs 80,176
average amount is almost identical: 2292.90 vs 2302.89
So the behavior difference is:

repeat users transact much more often
but they do not spend a different amount per transaction

Therefore:

the segment that behaves differently is repeat users
the difference is in transaction volume/frequency, not in average transaction size

##### TIME-BASED ANALYSIS:

In [122]:
#Exact Hour 
transactions_df['hour'] = pd.to_datetime(transactions_df['timestamp']).dt.hour


In [123]:
##transaction count by hour
hourly_volume = transactions_df.groupby('hour')['txn_id'].count()
hourly_volume

hour
0     8430
1     8226
2     8411
3     8267
4     8453
5     8341
6     8274
7     8363
8     8517
9     8313
10    8462
11    8311
12    8329
13    8264
14    8355
15    8300
16    8356
17    8177
18    8259
19    8397
20    8196
21    8355
22    8404
23    8240
Name: txn_id, dtype: int64

In [124]:
# Failure rate by hour -
transactions_df.groupby('hour')['status'].value_counts(normalize=True)

hour  status 
0     success    0.909134
      failed     0.090866
1     success    0.908826
      failed     0.091174
2     success    0.911426
      failed     0.088574
3     success    0.910487
      failed     0.089513
4     success    0.907962
      failed     0.092038
5     success    0.907685
      failed     0.092315
6     success    0.912014
      failed     0.087986
7     success    0.917733
      failed     0.082267
8     success    0.912763
      failed     0.087237
9     success    0.914351
      failed     0.085649
10    success    0.909714
      failed     0.090286
11    success    0.907833
      failed     0.092167
12    success    0.914155
      failed     0.085845
13    success    0.907551
      failed     0.092449
14    success    0.912029
      failed     0.087971
15    success    0.912892
      failed     0.087108
16    success    0.910603
      failed     0.089397
17    success    0.910358
      failed     0.089642
18    success    0.907737
      failed     0.09226

In [125]:
hourly = transactions_df.groupby('hour')['status'].value_counts(normalize=True).unstack(fill_value=0)
hourly['failure_rate'] = hourly['failed']
hourly[['failure_rate']]

status,failure_rate
hour,
0,0.090866
1,0.091174
2,0.088574
3,0.089513
4,0.092038
5,0.092315
6,0.087986
7,0.082267
8,0.087237


##### Experiment Analysis : 

In [126]:
## Success rate by experiment group
merged = transactions_df.merge(experiments_df, on="user_id", how="left")
success_rate = merged.groupby("variant")["status"].apply(lambda x: (x == "success").mean())


In [127]:
success_rate

variant
A    0.903578
B    0.904612
Name: status, dtype: float64

##### Summary:
I compared checkout UI variants A and B and found:

A success rate = 90.36%
B success rate = 90.46%


This is a checkout_ui_test A/B experiment.

Both groups show almost identical transaction success rates.

Variant B is only slightly higher than A by about 0.1 percentage points.


#### Conclusion:
There is no meaningful difference in success rate between the two checkout variants.

Variant B performed marginally better, but the difference is very small and likely not statistically significant.

#### 🧠 Business Context Analysis :

In [ ]:
## Do failures increase at night?
night_hours = list(range(20,24)) + list(range(0,6))
night_rate = hourly.loc[night_hours, 'failure_rate'].mean()

day_hours = list(range(9,18))
day_rate = hourly.loc[day_hours, 'failure_rate'].mean()

night_rate, day_rate


(0.10649875003459024, 0.08894600208428662)

Night has a higher failure rate than daytime.

Failures increase at night: about 10.7% at night vs 8.9% during the day.

Nighttime failure rate is roughly 1.7 percentage points higher than daytime.

In [129]:
## peak hours???

peak_hours = hourly_volume.sort_values(ascending=False).head(5)
print(peak_hours)

hour
8     8517
10    8462
4     8453
0     8430
2     8411
Name: txn_id, dtype: int64


#### Peak hour analysis
Peak transaction volume occurs at:

The busiest hour is 08:00, followed by 10:00.

These are the hours with the most user activity and transaction load.

#### Failure rate analysis

Failure rate is lowest during daytime/morning:
around 09:00–12:00 with rates near 8.6%
Failure rate spikes in the late evening:


20:00 = 12.86%,
21:00 = 13.33%,
22:00 = 13.02%,
23:00 = 12.84%


#### Summary
Peak volume hours are mostly morning and early day.

The worst failure performance is in the late evening.

So the system sees its highest transaction load earlier, while the highest failure risk is later at night.

## 6.Funnel analysis

#### User Behaviour

In [131]:
## Event distribution

events_df['event'].value_counts()

event
page_view           100000
initiate_payment     71730
enter_otp            62045
success              54556
Name: count, dtype: int64

In [132]:
## 🔥 Funnel counts

events_df.groupby('event')['user_id'].nunique()

event
enter_otp           35545
initiate_payment    38106
page_view           43294
success             33290
Name: user_id, dtype: int64

In [133]:
##  Count events by device

funnel = events_df.groupby(['device', 'event'])['user_id'].count().unstack(fill_value=0)
funnel

event,enter_otp,initiate_payment,page_view,success
device,,,,
android,35998,41707,59942,31532
ios,18084,20983,29974,15642
web,7963,9040,10084,7382


In [134]:
## conversion ratios

funnel['page_to_init'] = funnel['initiate_payment'] / funnel['page_view']
funnel['init_to_otp'] = funnel['enter_otp'] / funnel['initiate_payment']
funnel['otp_to_success'] = funnel['success'] / funnel['enter_otp']
funnel['page_to_success'] = funnel['success'] / funnel['page_view']

print(funnel[[
              'page_to_init',
              'init_to_otp',
              'otp_to_success',
              'page_to_success']])

event    page_to_init  init_to_otp  otp_to_success  page_to_success
device                                                             
android      0.695789     0.863117        0.875938         0.526042
ios          0.700040     0.861841        0.864964         0.521852
web          0.896470     0.880863        0.927038         0.732051


#### Summary 

Web has the best funnel completion across every stage.
android and ios are lower, with very similar performance to each other.

Stage-wise insight :

page_to_init =
web = 89.65%,
android = 69.58%,
ios = 70.00%,
Mobile users are much less likely to move from page view to payment initiation than web users.

init_to_otp =
web = 88.09%,
android = 86.31%,
ios = 86.18%,
Mobile users convert almost as well as web once payment is initiated.

otp_to_success =
web = 92.70%,
android = 87.59%,
ios = 86.50%,
Mobile users also lose more at the OTP-to-success step.

page_to_success =
web = 73.21%,
android = 52.60%,
ios = 52.19%,
Overall funnel completion is much higher on web than on mobile.



#### Conclusion
Mobile users show weaker funnel completion than web users.

 The biggest drop is at the first stage: fewer mobile users convert from page_view to initiate_payment, and they also drop more between enter_otp and success.

Web is the strongest channel, while Android and iOS perform similarly and are the weaker mobile funnel.

1. Payment failure increases at night
2. Mobile users show lower funnel completion
3. Revenue heavily dependent on gaming merchants

Pattern 1:

“Higher failure rate at night”

👉 Hypothesis:

UPI transactions fail more at night due to network congestion or bank downtime
Pattern 2:

“Mobile users drop more in funnel”

👉 Hypothesis:

Mobile UI/UX or OTP flow causes friction leading to drop-offs
Pattern 3:

“Repeat users generate same revenue”

👉 Hypothesis:

Repeat users are more comfortable with platform, but not leading to higher generate value..

## 7.HYPOTHESIS

#### Hypotheses :

🔹 H1: Night Failures
Payment failures increase at night due to UPI network congestion or bank downtime.

🔹 H2: Mobile Drop
Mobile funnel drop is caused by UI/UX friction during payment initiation and OTP steps.

🔹 H3: Repeat Users
Repeat users transact more frequently due to higher familiarity and trust in the platform.

#### Hypothesis Validation

🔹 H1 Validation


Failure rate by hour and payment method was analyzed.

Observation:
UPI shows higher failure rates during late hours compared to other methods.

Conclusion:
Hypothesis supported — UPI contributes significantly to night-time failures.

🔹 H2 Validation
Device-level funnel analysis shows mobile users have lower conversion at:
- page_view → initiate_payment
- otp → success

Conclusion:
Hypothesis supported — mobile experience is weaker than web.

🔹 H3 Validation
Repeat users have higher transaction counts but similar average transaction value.

Conclusion:
Hypothesis supported — difference is behavioral (frequency), not monetary.

## 8.SQL Analysis & Validation

### Funnel Conversion

In [135]:
query = """
SELECT 
    COUNT(DISTINCT CASE WHEN event='page_view' THEN user_id END) as page_view,
    COUNT(DISTINCT CASE WHEN event='initiate_payment' THEN user_id END) as initiate,
    COUNT(DISTINCT CASE WHEN event='enter_otp' THEN user_id END) as otp,
    COUNT(DISTINCT CASE WHEN event='success' THEN user_id END) as success
FROM events
"""
run_query(query)

,page_view,initiate,otp,success
0,43294,38106,35545,33290


In [136]:
conversion = (33290/43294)
conversion


0.7689287199149998

### Conversion by Device 

In [137]:
query = """
SELECT 
    device,
     COUNT(DISTINCT CASE WHEN event = 'page_view' THEN user_id END) AS page_view,
     COUNT(DISTINCT CASE WHEN event = 'initiate_payment' THEN user_id END) AS initiate_payment,
     COUNT(DISTINCT CASE WHEN event = 'enter_otp' THEN user_id END) AS enter_otp,
     COUNT(DISTINCT CASE WHEN event = 'success' THEN user_id END) AS success
FROM events
GROUP BY device """

run_query(query)

,device,page_view,initiate_payment,enter_otp,success
0,android,34871,28308,25609,23396
1,ios,22700,17148,15250,13472
2,web,9187,8324,7402,6880


In [138]:
query = """SELECT
  device,
  ROUND(
    100.0 * COUNT(DISTINCT CASE WHEN event = 'initiate_payment' THEN user_id END)
          / NULLIF(COUNT(DISTINCT CASE WHEN event = 'page_view' THEN user_id END), 0),
    2
  ) AS page_to_init_pct,
  ROUND(
    100.0 * COUNT(DISTINCT CASE WHEN event = 'enter_otp' THEN user_id END)
          / NULLIF(COUNT(DISTINCT CASE WHEN event = 'initiate_payment' THEN user_id END), 0),
    2
  ) AS init_to_otp_pct,
  ROUND(
    100.0 * COUNT(DISTINCT CASE WHEN event = 'success' THEN user_id END)
          / NULLIF(COUNT(DISTINCT CASE WHEN event = 'enter_otp' THEN user_id END), 0),
    2
  ) AS otp_to_success_pct,
  ROUND(
    100.0 * COUNT(DISTINCT CASE WHEN event = 'success' THEN user_id END)
          / NULLIF(COUNT(DISTINCT CASE WHEN event = 'page_view' THEN user_id END), 0),
    2
  ) AS page_to_success_pct
FROM events
GROUP BY device """

run_query(query)

,device,page_to_init_pct,init_to_otp_pct,otp_to_success_pct,page_to_success_pct
0,android,81.18,90.47,91.36,67.09
1,ios,75.54,88.93,88.34,59.35
2,web,90.61,88.92,92.95,74.89


Web users convert much better through the funnel than mobile users.

Mobile users lose the most at the first step (page_view → initiate_payment), especially iOS.

Overall conversion for mobile is substantially lower than web.


Web users convert much better through the funnel than mobile users.
Mobile users lose the most at the first step (page_view → initiate_payment), especially iOS.
Overall conversion for mobile is substantially lower than web.

### Failure Rate by Hour

In [139]:
query = """
SELECT 
    strftime('%H', timestamp) as hour,
    COUNT(CASE WHEN status='failed' THEN 1 END)*100.0/COUNT(*) as failure_rate,
    COUNT(CASE WHEN status='success' THEN 1 END)*100.0/COUNT(*) as success_rate
FROM transactions
GROUP BY hour
ORDER BY hour
"""
run_query(query)

,hour,failure_rate,success_rate
0,00,9.086595,90.913405
1,01,9.117433,90.882567
2,02,8.857449,91.142551
3,03,8.951252,91.048748
4,04,9.203833,90.796167
5,05,9.231507,90.768493
6,06,8.798646,91.201354
7,07,8.226713,91.773287
8,08,8.723729,91.276271
9,09,8.564898,91.435102


### Failure by Payment Method

In [140]:
query = """
SELECT 
    payment_method,
    COUNT(CASE WHEN status='failed' THEN 1 END)*100.0/COUNT(*) as failure_rate
FROM transactions
GROUP BY payment_method
"""
run_query(query)

,payment_method,failure_rate
0,UPI,9.120973
1,card,12.063445
2,wallet,4.944697


### Revenue by Merchant Category

In [141]:
query = """
SELECT 
    merchant_category,
    SUM(amount) as revenue
FROM transactions
GROUP BY merchant_category
ORDER BY revenue DESC
"""
run_query(query)

,merchant_category,revenue
0,gaming,2.046057e+08
1,fintech,1.276956e+08
2,ecommerce,7.640785e+07
3,edtech,5.067151e+07


### Repeat vs New Users

In [142]:
query = """
SELECT 
    is_repeat_user,
    COUNT(*) as total_txn,
    AVG(amount) as avg_amount
FROM transactions
GROUP BY is_repeat_user
"""
run_query(query)

,is_repeat_user,total_txn,avg_amount
0,0,80176,2302.885104
1,1,119824,2292.900930


## 9.Key Insights

1. gaming is the top revenue driver, far ahead of fintech, ecommerce, and edtech.
2. wallet has the lowest failure rate, while card has the highest failure rate and UPI is moderate.
3. Failure rates spike in the late evening (20:00–23:00), while daytime hours remain stable around 8.5%–9.2%.
4. Web has the strongest funnel conversion: page_to_success ~ 74.9%.
5. Mobile is weaker: android ~ 67.1% and iOS ~ 59.4%, with the largest mobile drop at page_view → initiate_payment.
6. Repeat users perform more transactions but do not spend more per transaction, 
   indicating higher engagement rather than higher spending.

##  10.Recommendations

- Improve late-evening reliability
  - Investigate and strengthen payment routing during `20:00–23:00`
  - Add retry/fallback logic for UPI and card payments during night hours

- Optimize mobile checkout
  - Simplify the mobile `page_view → initiate_payment` flow
  - Improve OTP handling on Android/iOS to reduce drop-off
  - Test and fix mobile UI/UX friction points in the payment funnel

- Focus on high-value merchant categories
  - Prioritize support and growth for `gaming`, the top revenue driver
  - Ensure gaming merchants have smooth checkout and lower failure exposures

- Reduce dependency on high-failure methods
  - Monitor and improve `card` failure performance
  - Promote `wallet` where feasible, since it has the lowest failure rate

- Strengthen repeat-user engagement
  - Use incentives or loyalty programs to keep repeat users active
  - Since repeat users transact more often, improve retention rather than pricing alone

## 11.Experiment Design & Analysis

### 1. Business Problem

Mobile users show lower funnel conversion compared to web users.
We want to test whether a new checkout experience can improve success rate.

### 2. Objective

Measure whether checkout variant B improves payment success rate compared to variant A.

### 3. Hypothesis

If users are exposed to checkout variant B, then payment success rate will increase compared to variant A.

### 4. Metrics

Primary Metric:
- Success Rate

Supporting Metrics:
- Page → Success conversion
- Page → Initiate conversion
- Initiate → OTP conversion
- OTP → Success conversion

### 5. Experiment Groups

- Control Group: Variant A
- Test Group: Variant B

### 6. Audience

All users included in the experiment dataset.

In [143]:
## SUCCESS RATE BY VARIANT

query = """
SELECT
  variant,
  ROUND(100.0 * SUM(CASE WHEN status='success' THEN 1 ELSE 0 END) / COUNT(*), 2) AS success_rate
FROM transactions
JOIN experiments USING(user_id)
GROUP BY variant
"""
run_query(query)


,variant,success_rate
0,A,90.36
1,B,90.46


### Result: Success Rate

Variant A = 90.36%  
Variant B = 90.46%

FUNNEL CONVERSION BY VARIANT :
- PAGE → SUCCESS

In [147]:
query = """
SELECT
  e.variant,
  ROUND(
    100.0 * COUNT(DISTINCT CASE WHEN event='success' THEN user_id END)
          / NULLIF(COUNT(DISTINCT CASE WHEN event='page_view' THEN user_id END), 0),
    2
  ) AS page_to_success_pct
FROM events e
JOIN experiments USING(user_id)
GROUP BY e.variant
"""
run_query(query)

,variant,page_to_success_pct
0,A,64.22
1,B,68.64


### Funnel Conversion (Page → Success)

Variant A = 64.22%

Variant B = 68.64%

In [150]:
query = """
SELECT
  e.variant,
  COUNT(DISTINCT CASE WHEN event='page_view' THEN user_id END) as page_view,
  COUNT(DISTINCT CASE WHEN event='initiate_payment' THEN user_id END) as initiate,
  COUNT(DISTINCT CASE WHEN event='enter_otp' THEN user_id END) as otp,
  COUNT(DISTINCT CASE WHEN event='success' THEN user_id END) as success
FROM events e
JOIN experiments USING(user_id)
GROUP BY e.variant
"""
run_query(query)

,variant,page_view,initiate,otp,success
0,A,31688,25592,23143,20349
1,B,31660,25605,23157,21730


#### Interpretation:

1.Variant B is performing better than A

2.success_rate is slightly higher for B: 90.46% vs 90.36%. 

3.The funnel conversion from page_view to success is noticeably better for B: 68.64% vs 64.22%. 

Summary:

-Both variants see almost equal page views and funnel counts, so the comparison is fair

-Variant B converts more users all the way to success. 

-In other words, B is more efficient at turning page views into complete  payments. 

#### statistical testing :

1. Does variant B improve checkout success compared to variant A?

2. metrics :

Primary metric: success_rate

    counts from data:

    success_A = 20349

    total_A = 31688

    success_B = 21730

    total_B = 31660

3. the hypotheses:

H0 (null): A and B have the same success rate

H1 (alternative): B has a higher success rate than A

4. Using a two-proportion z-test





In [160]:
from statsmodels.stats.proportion import proportions_ztest

success_A = 20349
total_A = 31688

success_B = 21730
total_B = 31660

success = [success_B, success_A]
nobs = [total_B, total_A]

z_stat, p_value = proportions_ztest(success, nobs, alternative='larger')

success_rate_A = success_A / total_A
success_rate_B = success_B / total_B
lift = success_rate_B - success_rate_A

print("Variant A success rate:", round(success_rate_A * 100, 2), "%")
print("Variant B success rate:", round(success_rate_B * 100, 2), "%")
print("Absolute lift:", round(lift * 100, 2), "percentage points")
print("z-statistic:", round(z_stat, 3))
print("p-value:", p_value)

Variant A success rate: 64.22 %
Variant B success rate: 68.64 %
Absolute lift: 4.42 percentage points
z-statistic: 11.775
p-value: 2.6233437812878437e-32


I have tested checkout variants A and B on the page-to-success funnel.

 Variant B has a higher conversion rate (68.64% vs 64.22%), delivering a 4.42 point lift.

  The two-proportion z-test gives a very small p-value (2.62e-32), so the result is statistically significant. 
  
  Based on this analysis, Variant B should be preferred over Variant A.

## 12.KPI Framework

Primary KPIs:
- Payment Success Rate = Successful transactions / Total transactions
- Funnel Conversion Rate = Users completing payment / Users entering funnel

Secondary KPIs:
- Average Transaction Value (ATV)
- Transactions per User

Segment KPIs:
- Conversion Rate by Device (Mobile vs Web)
- Success Rate by Payment Method (UPI, Card, Wallet)
- Engagement by User Type (Repeat vs New)

## 13.**Business Impact**

1.**Mobile page-to-success conversion is only ~52% versus ~73% for web, so improving mobile funnel could recover roughly 20 percentage points of lost completion on the larger mobile audience.**

2.**Late-hour failure rate rises to ~10.7% versus ~8.9% in daytime, meaning night failures are about 17% higher and reducing that gap would improve overall success rate.**

3.**Checkout Variant B delivered a 4.42 percentage-point lift in page→success conversion, with a highly significant p-value of ~2.6e-32.**

4.**Repeat users generate 119,824 transactions versus 80,176 for new users, so increasing repeat-user engagement can drive higher transaction volume without raising average spend.**

5.**Even a 5% improvement in conversion rate can lead to a substantial increase in revenue at scale.**

## 14.Limitations

- Dataset is simulated and may not fully capture real-world complexities
- External factors such as bank downtime or network issues are assumed but not directly measured
- Funnel steps are inferred from event data and may not represent exact user journeys
- No statistical significance testing was performed for experiment results

## 15.Final Summary

This project performed end-to-end product analytics on a payments platform dataset.

Starting with data exploration and quality checks, key behavioral patterns were identified
in user funnel performance and payment success rates.

This analysis identified late-night failure risk, mobile funnel weakness, and a strong experiment lift for variant B.

Hypotheses were formed and validated using SQL-based analysis, revealing issues
in mobile conversion and time-based failure patterns.

An experiment framework was designed to improve checkout performance,
and The strongest recommendation is to adopt Variant B while continuing to refine mobile checkout and monitor payment failure patterns.

This analysis demonstrates how data-driven insights can be used to optimize product performance
and business outcomes.